In [28]:
import os
os.path.exists('../../../SBT-data/InterFuser/2025-02-08|14:47:39')

True

In [39]:
a = np.array([])
a = None

In [40]:
a is None

True

In [37]:
if a.isempty():
    print(a)

AttributeError: 'numpy.ndarray' object has no attribute 'isempty'

In [1]:
from SBT.simulator_utils import run_carla, kill_carla, kill_by_port
kill_by_port(2000)


fuser -k 2000/tcp
Stop by port
Fail to stop by port: 1


In [ ]:
import time
from SBT.simulator_utils import run_carla, kill_carla, kill_by_port

# run_carla(rander=True)
# time.sleep(10)
for i in range(10):
    kill_by_port(2000+i)



fuser -k 2000/tcp
Stop by port

fuser -k 2001/tcp
Stop by port
Fail to stop by port: 1

fuser -k 2002/tcp
Stop by port
Fail to stop by port: 1

fuser -k 2003/tcp
Stop by port
Fail to stop by port: 1

fuser -k 2004/tcp
Stop by port
Fail to stop by port: 1

fuser -k 2005/tcp
Stop by port
Fail to stop by port: 1

fuser -k 2006/tcp
Stop by port
Fail to stop by port: 1

fuser -k 2007/tcp
Stop by port
Fail to stop by port: 1

fuser -k 2008/tcp
Stop by port
Fail to stop by port: 1

fuser -k 2009/tcp
Stop by port
Fail to stop by port: 1


: 

In [4]:
!lsof -t -i:2002

[1,2,3,4,5]

In [8]:
import os
import sys
import numpy as np
import pandas as pd


def get_folder(folder, indexs):
    result = []
    dirs = os.listdir(folder)
    dirs.sort()
    dirs = dirs[3:-1]
    for i in indexs:
        result.append(dirs[i])
    return result

def get_max_gear(case):
    return pd.read_csv(case+'control.csv')['gear'].max()


def get_fitness(criterion_dir, fitness_dir, col, length=False, direction=True):
    folder = criterion_dir.replace('criterion.csv','')

    with open(criterion_dir, 'r') as f:
        total_lines = sum(1 for _ in f)

    criterion_header = ["RouteCompletionTest",   
                "RouteCompletionTest_figure",
                "OutsideRouteLanesTest", 
                "OutsideRouteLanesTest_figure",
                "CollisionTest",         
                "CollisionTest_figure",
                "RunningRedLightTest",   
                "RunningRedLightTest_figure",
                "RunningStopTest",       
                "RunningStopTest_figure",
                "InRouteTest", 
                "InRouteTest_figure",          
                "AgentBlockedTest",
                "AgentBlockedTest_figure",      
                "Timeout"]
    
    fitness_header = ["DOL","DVE","DPD","DSM","DFD"]
    if direction:
        fitness_header = ["DOL","DVE",'DVE-d',"DPD","DSM","DFD"]

    if length:
        criterion = pd.read_csv(criterion_dir,names=criterion_header, skiprows=range(total_lines-length))
        fitness = pd.read_csv(fitness_dir,names=fitness_header, skiprows=range(total_lines-length))
    else:
        criterion = pd.read_csv(criterion_dir,names=criterion_header)
        fitness = pd.read_csv(fitness_dir,names=fitness_header)

    if direction:
        fitness['DVE'] = fitness['DVE'].apply(lambda data: float(data[1:]))
        fitness['DVE-d'] = fitness['DVE-d'].apply(lambda data: float(data[:-1]))
    else:
        fitness['DVE-d'] = np.zeros_like(fitness['DVE'])
    result = pd.DataFrame()

    result['RouteCompletionTest']   =   criterion["RouteCompletionTest_figure"]/100
    result['OutsideRouteLanesTest'] = 1-criterion["OutsideRouteLanesTest_figure"]/100
    result['CollisionTest']         =   criterion["CollisionTest"]/2*2
    result['RunningRedLightTest']   = 1-criterion["RunningRedLightTest"]
    result['RunningStopTest']       = 1-criterion["RunningStopTest"]
    result['InRouteTest']           = 1-criterion["InRouteTest"]
    result['AgentBlockedTest']      = 1-criterion["AgentBlockedTest"]
    result['Timeout']               = 1-criterion["Timeout"]

    DVE = fitness['DVE'].copy()/2
    DVE[fitness['DVE'] >= 2] = 1
    collisionTest = result['CollisionTest'].copy()
    collisionTest[result['CollisionTest']==0] = DVE[result['CollisionTest']==0]
    collisionTest[result['CollisionTest']==1] = 0
    result.loc[:,'CollisionTest'] = collisionTest
    result.loc[:,'CollisionDirection'] = fitness['DVE-d']

    DOL = fitness['DOL'].copy() - 0.5
    DOL[fitness['DOL'] >= 1.5] = 1
    DOL[fitness['DOL'] <= 0.5] = 0
    result.loc[:,'OutsideRouteLanesTest'] = 1-DOL

    DPD = fitness['DPD'].copy()/10
    DPD[fitness['DPD'] >= 10] = 1
    result['PedestrianTest'] = DPD

    change = []
    for i, case in enumerate(get_folder(folder, result[result['CollisionTest'] == 0].index.to_numpy())):
        if get_max_gear(folder+case+'/') == 0:
            index = result[result['CollisionTest'] == 0].index.to_numpy()[i]
            change.append(index)
            print('ERROR: Wrong Collision -', folder+case)

    for index in change:
        result.loc[index,'CollisionTest'] = 1
        result.loc[index,'RouteCompletionTest'] = 1
        # result['RouteCompletionTest'][index] = 1
        # result['CollisionTest'][index] = 1

    return result[col].to_numpy()


In [10]:
root = '/home/guannan/Projects/SBT-data/InterFuser/2025-01-13|01:28:07--SAVE_IMG/'
result = get_fitness(root+'criterion.csv', root+'fitness.csv', col='CollisionTest', length=7)

In [13]:
result.shape, result.dtype, result 

((7,),
 dtype('float64'),
 array([0.50378579, 0.        , 0.48899263, 0.35936496, 0.51451778,
        0.43339974, 0.        ]))